In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.append(str(Path().resolve().parent / 'src'))

In [2]:
# Funciones para crear señales 
from utils import ema_price_signal, macd_signal, stochastic_oversold_signal, ichimoku_buy_signals, ichimoku_signal_aggressive, ichimoku_signal_kijun_cross, ichimoku_signal_chikou_break, bollinger_reversion_signal
from utils import generar_estrategias_combinadas
#Funcion para crear Target
from utils import crear_target

In [3]:
df = pd.read_csv('../data/VOO.csv')

In [4]:
df.head(3)

,date,close,high,low,open,volume,daily_return,rsi,macd,macd_signal,...,ema_200,ema_9,ichimoku_a,ichimoku_b,ichimoku_base,ichimoku_conversion,adx,volume_sma_20,volume_ratio,awesome_osc
0,2011-06-23,91.127,91.158,89.689,90.338,260750,-0.003,44.615,-0.773,-0.887,...,89.286,91.098,91.583,93.16,92.395,90.771,21.804,116412.5,2.240,-2.011
1,2011-06-24,90.125,91.181,89.969,91.150,114500,-0.011,39.854,-0.802,-0.870,...,89.295,90.904,91.583,93.16,92.395,90.771,21.981,119115.0,0.961,-1.842
2,2011-06-27,90.886,91.228,89.985,90.187,77850,0.008,44.682,-0.756,-0.847,...,89.310,90.900,91.564,93.16,92.356,90.771,22.092,120627.5,0.645,-1.670


In [5]:
df['date'] = pd.to_datetime(df['date'])

In [6]:
# Creación de Características Numéricas Avanzadas 
df['body_size'] = abs(df['close'] - df['open'])
df['upper_wick'] = df['high'] - df[['open', 'close']].max(axis=1)
df['lower_wick'] = df[['open', 'close']].min(axis=1) - df['low']

# Calcula el ROC, que mide el cambio porcentual del activo a lo larco del tiempo 
df['rsi_roc_5'] = df['rsi'].pct_change(periods=5) * 100

# Características Cíclicas del Tiempo
# Se crean características cíclicas del tiempo usando transformaciones sinusoidales
# Esto convierte el mes en una característica cíclica continua (sin saltos entre diciembre y enero)
# month_sin y month_cos capturan la naturaleza cíclica anual del mercado
# Usamos seno y coseno para capturar la naturaleza cíclica de los meses
df['month_sin'] = np.sin(2 * np.pi * df['date'].dt.month / 12)
df['month_cos'] = np.cos(2 * np.pi * df['date'].dt.month / 12)

# Este enfoque resuelve varios problemas:
# 1. Continuidad: Diciembre (12) y Enero (1) están cerca en tiempo
#    pero aparecerían lejos si usáramos números crudos
# 2. Periodicidad: La transformación mantiene la relación cíclica
#    donde el mes 12 vuelve al mes 1
# 3. Distancia: La codificación seno-coseno preserva la verdadera
#    distancia temporal entre meses de forma circular

# Esto crea los cruces entre el precio, kinun y tenkan 
df['price_vs_kijun'] = df['close'] - df['ichimoku_base']
df['tenkan_vs_kijun'] = df['ichimoku_conversion'] - df['ichimoku_base']

# Limpiar los NaNs generados por los nuevos cálculos
df = df.dropna()

### SEÑALES

In [7]:
df = ema_price_signal(df)
df = macd_signal(df)
df = stochastic_oversold_signal(df)
df = ichimoku_buy_signals(df)
df = ichimoku_signal_aggressive(df)
df = ichimoku_signal_kijun_cross(df)
df = ichimoku_signal_chikou_break(df)
df = bollinger_reversion_signal(df)

In [8]:
combinaciones = [
    ('signal_macd_buy', 'signal_stochastic_buy'),
    ('signal_ema_price', 'signal_macd_buy'),
    ('signal_ichimoku_kijun_cross', 'signal_stochastic_buy'),
    ('signal_ema_price', 'signal_stochastic_buy')]

df, nuevas_estrategias = generar_estrategias_combinadas(df, combinaciones)

 Creando Estrategias Combinadas
Señal combinada 'signal_macd_&_stochastic' creada.
Señal combinada 'signal_ema_price_&_macd' creada.
Señal combinada 'signal_ichimoku_kijun_cross_&_stochastic' creada.
Señal combinada 'signal_ema_price_&_stochastic' creada.
Se han añadido las siguientes columnas de señales combinadas al dataframe:
- signal_macd_&_stochastic
- signal_ema_price_&_macd
- signal_ichimoku_kijun_cross_&_stochastic
- signal_ema_price_&_stochastic


In [9]:
# Aplicar la función para crear un target viable: aqui dejamos un target en el que le indicamos si el precio sube un 4% en 19 dias. 
df = crear_target(df, dias_futuro=19, umbral_retorno=0.04)

In [10]:
#exportamos los datos a un csv en la carpeta data
df.to_csv('../data/VOO_ind_signal.csv', index=False)

In [11]:
df.head()

,date,close,high,low,open,volume,daily_return,rsi,macd,macd_signal,...,signal_ichimoku_aggressive,signal_ichimoku_kijun_cross,signal_ichimoku_chikou,signal_bollinger_buy,signal_macd_&_stochastic,signal_ema_price_&_macd,signal_ichimoku_kijun_cross_&_stochastic,signal_ema_price_&_stochastic,retorno_futuro,target
5,2011-06-30,93.761,93.885,93.092,93.186,124850,0.010,59.020,-0.219,-0.626,...,False,False,False,False,False,False,False,False,-0.012926,0
6,2011-07-01,95.175,95.252,93.652,93.807,139550,0.015,64.322,0.067,-0.488,...,False,False,False,False,False,False,False,False,-0.035104,0
7,2011-07-05,95.019,95.252,94.786,95.097,91850,-0.002,63.348,0.277,-0.335,...,False,False,False,False,False,False,False,False,-0.036961,0
8,2011-07-06,95.159,95.283,94.600,94.848,94850,0.001,63.877,0.450,-0.178,...,False,False,False,False,False,False,False,False,-0.062380,0
9,2011-07-07,96.200,96.418,95.858,96.014,195400,0.011,67.616,0.664,-0.009,...,False,False,False,False,False,False,False,False,-0.067838,0
